# SayIt: Deteksi Zona Tanggal Kedaluwarsa

Model **kedua** yang terpisah dari model uang: satu kelas saja (`expired_date_zone`),
tugasnya hanya menemukan **letak** tanggal pada kemasan, bukan membacanya. Pembacaan
tetap dikerjakan EasyOCR dan `sayit/vision/date_parser.py`.

**Kenapa perlu.** Tanpa model ini, `ExpiryReader` memakai jalur cadangan FR-12: OCR
seluruh gambar. Terukur di laptop tim: jalur zona **0,46 detik**, jalur cadangan
**4,99 detik** untuk jawaban yang sama, dan jalur cadangan selalu menyatakan diri
"kurang yakin" sesuai PRD 8.8 langkah 6.

**Kenapa dataset publik boleh dipakai di sini**, padahal untuk model uang bermasalah:
yang dipelajari hanya **kotak lokasi**, bukan isi tanggalnya. Perbedaan konvensi
penulisan tanggal antar-negara (`03/12` = 3 Desember di Indonesia, 12 Maret di Amerika)
tidak berpengaruh sama sekali terhadap deteksi letak.

**Target PRD Bagian 11:** mAP@0.5 >= 0,75.


## 1. Persiapan

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics roboflow imagehash

In [ ]:
import random, shutil
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image as IPImage, display
from PIL import Image
from roboflow import Roboflow
from ultralytics import YOLO

DRIVE_DIR = Path('/content/drive/MyDrive/sayit')
RUNS_DIR = DRIVE_DIR / 'runs'
MODELS_DIR = DRIVE_DIR / 'models'
DATA_DIR = DRIVE_DIR / 'datasets'
for d in (RUNS_DIR, MODELS_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Konfigurasi

Dataset di bawah ini hasil penelusuran Roboflow Universe. **Periksa sendiri isinya
sebelum dipakai**: buka tautannya, lihat contoh gambarnya, dan pilih versi
**tanpa augmentasi** (augmentasi akan ditambahkan saat training, bukan di dataset).

| nama | slug | jumlah | catatan |
| :--- | :--- | ---: | :--- |
| CoolSTuff | `expiry-date-detection-i3bav` | 3.797 | terbesar, mAP@50 yang dilaporkan 80,3% |
| Abhilasha | `expiry-date-detection-ssxnm` | 5.200 | kelasnya bernama `date` |
| ML Model | `expiry-date-recognition` | 2.500 | |
| CHOI | `expiry-date` | 320 | kecil, untuk pelengkap |

Nama kelas berbeda-beda antar-dataset (`date`, `Expiry-date`, `expiration_date`).
Karena modelnya **satu kelas**, semua nama itu dilebur menjadi satu; tidak ada risiko
label tertukar seperti pada model uang.


In [ ]:
# ====== ISI BAGIAN INI ======
# Isi `workspace` dan `versi` sesuai yang kamu lihat di halaman Roboflow.
# Hapus tanda komentar pada dataset yang mau dipakai.
DATASETS = [
    dict(nama='COOL', workspace='coolstuff-rfnc5', project='expiry-date-detection-i3bav', versi=1),
    # dict(nama='ABHI', workspace='abhilashasacademicworkspace-8igt2',
    #      project='expiry-date-detection-ssxnm', versi=1),
    # dict(nama='MLM',  workspace='ml-model-tlmqd', project='expiry-date-recognition', versi=1),
    # dict(nama='CHOI', workspace='choi-t72ze', project='expiry-date', versi=1),

    # Foto kemasan sendiri yang sudah dilabeli. SELURUHNYA menjadi split test,
    # supaya angka mAP-nya mengukur pemakaian nyata dan bukan kemiripan dengan data latih.
    # dict(nama='SENDIRI', lokal='/content/drive/MyDrive/sayit/data/zona_sendiri', sendiri=True),
]

KELAS = 'expired_date_zone'      # satu kelas; nama apa pun dari dataset dilebur ke sini

SPLIT_RATIO = (0.7, 0.2, 0.1)    # train, val, test. Bagian test diabaikan bila ada dataset sendiri
SEED = 42
PHASH_MAX_DIST = 8

MODEL_WEIGHTS = 'yolov8n.pt'
EPOCHS   = 100
PATIENCE = 20
IMGSZ    = 640
BATCH    = 16
RUN_NAME = 'zone_v1'

# Augmentasi. Pelajaran dari model uang: bawaan ultralytics `degrees=0.0` membuat
# model tidak pernah melihat objek miring, dan pengguna tunanetra tidak bisa membidik.
AUG = dict(
    degrees=180,         # kemasan bisa dipegang dengan sudut apa pun
    perspective=0.0005,
    scale=0.6,
    translate=0.2,
    hsv_v=0.5,
    hsv_s=0.8,
    fliplr=0.5,          # berbeda dari model uang: tanggal cermin tetap berupa tanggal,
                         # dan yang dicari hanya letaknya, bukan bacaannya
)
# ============================

random.seed(SEED)
np.random.seed(SEED)

def ambil_kunci_roboflow() -> str:
    """Ambil API key Roboflow, tahan terhadap perbedaan versi Colab.

    Sebagian versi `google.colab.userdata.get()` mengembalikan str, sebagian
    mengembalikan dict. Kalau keduanya gagal, kunci diminta lewat ketikan.
    """
    kunci = None
    try:
        from google.colab import userdata
        kunci = userdata.get('ROBOFLOW_API_KEY')
    except Exception as e:
        print('Secret ROBOFLOW_API_KEY tidak terbaca:', type(e).__name__)
    if isinstance(kunci, dict):
        kunci = (kunci.get('ROBOFLOW_API_KEY') or kunci.get('value')
                 or next((v for v in kunci.values() if isinstance(v, str)), None))
    if not isinstance(kunci, str) or not kunci.strip():
        from getpass import getpass
        kunci = getpass('Tempel Roboflow API key (Settings > API Keys di roboflow.com): ')
    return kunci.strip()


RF_KEY = ambil_kunci_roboflow()
assert RF_KEY, 'API key kosong.'
rf = Roboflow(api_key=RF_KEY)

## 3. Fungsi bantu

In [ ]:
def image_files(root: Path):
    for split in ('train', 'valid', 'val', 'test'):
        d = root / split / 'images'
        if d.exists():
            for p in sorted(d.iterdir()):
                if p.is_file():
                    yield p


def read_boxes(lbl: Path):
    """Semua kotak, kelas diabaikan. Model ini hanya punya satu kelas."""
    kotak = []
    if not lbl.exists():
        return kotak
    for baris in lbl.read_text().splitlines():
        bagian = baris.split()
        if len(bagian) < 5:
            continue
        nilai = list(map(float, bagian[1:]))
        if len(nilai) != 4:                       # sebagian dataset menyimpan poligon
            xs, ys = nilai[0::2], nilai[1::2]
            nilai = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2,
                     max(xs) - min(xs), max(ys) - min(ys)]
        if all(0 <= v <= 1 for v in nilai) and nilai[2] > 0 and nilai[3] > 0:
            kotak.append(nilai)
    return kotak


def phash_bits(p: Path):
    with Image.open(p) as im:
        return imagehash.phash(im.convert('RGB')).hash.flatten()

## 4. Unduh dan kumpulkan

Sel ini juga mencetak berapa gambar yang **tidak punya kotak sama sekali**. Gambar
seperti itu tetap berguna sebagai contoh latar (mengurangi deteksi palsu), tetapi kalau
jumlahnya sangat besar, datasetnya patut dicurigai.


In [ ]:
def siapkan(d):
    if d.get('lokal'):
        root = Path(d['lokal'])
        assert root.exists(), f"{d['nama']}: folder tidak ada: {root}"
        asal = f'lokal {root}'
    else:
        ds = rf.workspace(d['workspace']).project(d['project']).version(d['versi']).download(
            'yolov8', location=f"/content/zona_{d['nama']}", overwrite=True)
        root = Path(ds.location)
        asal = f"roboflow {d['project']}"
    n = sum(1 for _ in image_files(root))
    assert n, f"{d['nama']}: tidak ada gambar di {root}"
    nama_kelas = []
    if (root / 'data.yaml').exists():
        with open(root / 'data.yaml') as f:
            nama_kelas = yaml.safe_load(f).get('names', [])
    print(f"{d['nama']:10s} {asal:55s} {n:>5d} gambar, kelas asal {nama_kelas}")
    return root


records = []
log = Counter()
for d in DATASETS:
    root = siapkan(d)
    for img in image_files(root):
        kotak = read_boxes(img.parent.parent / 'labels' / f'{img.stem}.txt')
        records.append({'dataset': d['nama'], 'path': img, 'boxes': kotak,
                        'sendiri': bool(d.get('sendiri'))})
        log[f"{d['nama']} dipakai"] += 1
        if not kotak:
            log[f"{d['nama']} tanpa kotak (jadi contoh latar)"] += 1

for k, v in sorted(log.items()):
    print(f'{k:48s}: {v}')
print(f'\nTotal gambar: {len(records)}')
print(f'Total kotak  : {sum(len(r["boxes"]) for r in records)}')

punya_sendiri = [d['nama'] for d in DATASETS if d.get('sendiri')]
if not punya_sendiri:
    print('\nPERINGATAN: belum ada foto kemasan sendiri (sendiri=True). Split test akan')
    print('            berasal dari sumber yang sama dengan data latih, sehingga mAP-nya')
    print('            optimistis. Ukur yang sebenarnya di laptop dengan')
    print(r'            python scripts\ukur_kedaluwarsa.py')

## 5. Kelompokkan foto yang mirip

Dataset publik kerap memuat foto yang sama atau nyaris sama, apalagi bila beberapa
dataset digabung. Tanpa langkah ini, foto kembar bisa terpisah ke train dan test, dan
mAP-nya jadi bohong.


In [ ]:
H = np.stack([phash_bits(r['path']) for r in records])
N = len(records)
parent = list(range(N))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


for i in range(0, N, 200):
    d = (H[i:i + 200, None, :] != H[None, :, :]).sum(-1)
    for a, b in zip(*np.nonzero(d <= PHASH_MAX_DIST)):
        a += i
        if a < b:
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb

for i, r in enumerate(records):
    r['grup'] = find(i)
ukuran = Counter(Counter(r['grup'] for r in records).values())
print(f'{len(set(r["grup"] for r in records))} kelompok dari {N} gambar')
print('Sebaran ukuran kelompok:', dict(sorted(ukuran.items())))

## 6. Bagi train/val/test

Sama seperti notebook uang: bila ada foto sendiri, **seluruhnya** menjadi split test.


In [ ]:
by_group = defaultdict(list)
for r in records:
    by_group[r['grup']].append(r)

grup_sendiri = {g for g, rs in by_group.items() if any(r['sendiri'] for r in rs)}
ada_sendiri = bool(grup_sendiri)

sisa = [g for g in by_group if g not in grup_sendiri]
rng = random.Random(SEED)
rng.shuffle(sisa)
n = len(sisa)
n_test = 0 if ada_sendiri else round(n * SPLIT_RATIO[2])
n_val = round(n * SPLIT_RATIO[1])

split_of_group = {g: 'test' for g in grup_sendiri}
for k, g in enumerate(sisa):
    split_of_group[g] = 'test' if k < n_test else ('val' if k < n_test + n_val else 'train')
for r in records:
    r['split'] = split_of_group[r['grup']]

per = defaultdict(set)
for r in records:
    per[r['grup']].add(r['split'])
assert all(len(s) == 1 for s in per.values()), 'ada kelompok yang terpecah antar-split'
print('Tidak ada kelompok kemiripan yang terpecah antar-split.')

if ada_sendiri:
    asing = [r for r in records if r['split'] == 'test' and not r['sendiri']]
    assert not asing, f'{len(asing)} gambar bukan foto sendiri masuk split test'
    print(f"Split test berisi HANYA foto sendiri: {sum(r['split'] == 'test' for r in records)} gambar.")

baris = []
for split in ('train', 'val', 'test'):
    rs = [r for r in records if r['split'] == split]
    baris.append({'split': split, 'gambar': len(rs),
                  'kotak': sum(len(r['boxes']) for r in rs),
                  'tanpa_kotak': sum(not r['boxes'] for r in rs),
                  **{f"dari_{d['nama']}": sum(r['dataset'] == d['nama'] for r in rs)
                     for d in DATASETS}})
split_stats = pd.DataFrame(baris).set_index('split')
display(split_stats)

## 7. Tulis dataset satu kelas

In [ ]:
V1_ROOT = Path('/content/data_zona')
if V1_ROOT.exists():
    shutil.rmtree(V1_ROOT)
for split in ('train', 'val', 'test'):
    (V1_ROOT / split / 'images').mkdir(parents=True)
    (V1_ROOT / split / 'labels').mkdir(parents=True)

for i, r in enumerate(records):
    stem = f"{r['dataset']}_{i:05d}"
    shutil.copy(r['path'], V1_ROOT / r['split'] / 'images' / f"{stem}{r['path'].suffix}")
    # indeks kelas selalu 0: modelnya satu kelas
    isi = '\n'.join('0 ' + ' '.join(f'{v:.6f}' for v in b) for b in r['boxes'])
    (V1_ROOT / r['split'] / 'labels' / f'{stem}.txt').write_text(isi)

V1_YAML = V1_ROOT / 'data.yaml'
with open(V1_YAML, 'w') as f:
    yaml.safe_dump({'path': str(V1_ROOT),
                    'train': str(V1_ROOT / 'train' / 'images'),
                    'val': str(V1_ROOT / 'val' / 'images'),
                    'test': str(V1_ROOT / 'test' / 'images'),
                    'names': {0: KELAS}}, f, sort_keys=False)
split_stats.to_csv(DATA_DIR / 'zona_split_stats.csv')
print('Dataset zona ditulis ke', V1_ROOT)

### 7.1 Cek visual

Selalu lihat beberapa contoh sebelum melatih. Kotak yang meleset atau anotasi yang
menandai seluruh kemasan (bukan tanggalnya saja) langsung terlihat di sini.


In [ ]:
import cv2
import matplotlib.pyplot as plt

contoh = random.Random(SEED).sample(
    [r for r in records if r['boxes']], k=min(8, sum(1 for r in records if r['boxes'])))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.flatten(), contoh):
    im = cv2.imread(str(r['path']))[:, :, ::-1].copy()
    h, w = im.shape[:2]
    for xc, yc, bw, bh in r['boxes']:
        x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
        x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 220, 0), 3)
    ax.imshow(im)
    ax.set_title(f"{r['dataset']} - {len(r['boxes'])} kotak", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Training

`fliplr=0.5` sengaja **berbeda** dari notebook uang. Pada uang, tulisan nominal yang
tercermin membingungkan model. Di sini yang dicari hanya letak kotak, dan tanggal yang
tercermin tetap berupa blok teks kecil, jadi pencerminan justru menambah variasi.


In [ ]:
model = YOLO(MODEL_WEIGHTS)
model.train(
    data=str(V1_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED,
    project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True, save_period=10, plots=True,
    **AUG,
)
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
print('Augmentasi yang dipakai:', AUG)

## 9. Evaluasi

In [ ]:
model = YOLO(str(BEST_PT))
m = model.val(data=str(V1_YAML), split='test', project=str(RUNS_DIR),
              name=RUN_NAME + '_test', exist_ok=True)
print(f'mAP@0.5 = {m.box.map50:.3f} | mAP@0.5:0.95 = {m.box.map:.3f}')
print(f'precision = {float(m.box.p[0]):.3f} | recall = {float(m.box.r[0]):.3f}')
print(f'\nTarget PRD Bagian 11: mAP@0.5 >= 0,75 -> '
      f'{"TERCAPAI" if m.box.map50 >= 0.75 else "BELUM TERCAPAI"}')
if not ada_sendiri:
    print('\nCATATAN: split test sesumber dengan data latih, jadi angka di atas optimistis.')
    print('         Angka yang sah diukur di laptop dengan scripts\\ukur_kedaluwarsa.py')

## 10. Simpan bobot

Salin berkas hasil unduhan ke `models/best_zone.pt` di laptop. `config/settings.yaml`
sudah menunjuk path itu (`vision.expiry.zone_weights`), dan `ExpiryReader` otomatis
berpindah dari jalur cadangan ke jalur zona begitu berkasnya ada.

Sesudah itu ukur ulang dan bandingkan dengan patokan jalur cadangan:

```
python scripts\ukur_kedaluwarsa.py
```


In [ ]:
stamp = datetime.now().strftime('%Y%m%d_%H%M')
dst = MODELS_DIR / f'best_zone_{RUN_NAME}_{stamp}.pt'
shutil.copy(BEST_PT, dst)
print('Bobot disalin ke', dst)
print('Unduh berkas itu, lalu di laptop:')
print(r'   copy <berkas>.pt models\best_zone.pt')